# MultiDelete (ECCV 2024) trên ALBEF: Flickr30K và NLVR2, chạy từ A đến Z

Notebook này chỉ điều khiển; code nằm trong 4 file `.py` đặt cạnh notebook: `multidelete.py` (điều phối), `run_md_retrieval.py` (engine Flickr30K), `nlvr2_engine.py` (engine NLVR2), `demo_engine.py` (demo). Chi tiết, thời gian chạy và kết quả mẫu xem `README.md`.

**Cách dùng (Colab):** upload cả thư mục `MultiDelete_Unlearning` lên Google Drive, mở notebook này, *Runtime → Change runtime type → T4 GPU*, rồi chạy lần lượt từng cell từ trên xuống.

| Cell | Việc |
|---|---|
| §0 | Cấu hình, nối Drive, Kaggle |
| §1 | Cài thư viện, clone repo, vá lỗi |
| §2 | Chạy thử luồng bằng profile `smoke` |
| §3 | Flickr30K, toàn bộ |
| §4 | NLVR2, toàn bộ |
| §5 | Demo định tính (tùy chọn) |
| §6 | Xem bảng và hình kết quả |

Lần chạy nào đã có kết quả (`.json`) trong `SAVE` sẽ được bỏ qua, nên bị ngắt phiên thì mở lại, chạy lại §0 rồi cell đang dở.

## §0. Cấu hình

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/tnde0401/multimodal-unlearning-multidelete"
    CODE_DIR = "/content/multimodal-unlearning-multidelete"   # Colab: tự clone repo, không cần upload lên Drive
    if not os.path.isdir(CODE_DIR):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
    WORK = "/content/work"                                       # repo, dữ liệu, cache, checkpoint tạm (mất khi ngắt phiên)
    SAVE = "/content/drive/MyDrive/multidelete_out"              # kết quả: nằm trên Drive nên không mất
    try:                                                         # chỉ cần cho Flickr30K (tải ảnh từ Kaggle)
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception as e:
        print("Chưa có Colab Secrets KAGGLE_USERNAME / KAGGLE_KEY:", repr(e)[:120],
              "\n-> Đặt FLICKR_IMAGES bên dưới nếu bạn đã có ảnh Flickr30K.")
else:
    CODE_DIR = os.getcwd()
    WORK, SAVE = os.path.abspath("./work"), os.path.abspath("./results")

FLICKR_IMAGES = ""      # nếu đã có ảnh Flickr30K: đường dẫn thư mục chứa 1000092795.jpg (bỏ qua bước tải Kaggle)

for f in ("multidelete.py", "run_md_retrieval.py", "nlvr2_engine.py", "demo_engine.py"):
    assert os.path.isfile(f"{CODE_DIR}/{f}"), f"Thiếu {CODE_DIR}/{f}. Hãy upload đủ 4 file .py vào CODE_DIR."
os.chdir(CODE_DIR)
COMMON = f"--work {WORK} --save {SAVE}" + (f" --flickr-images {FLICKR_IMAGES}" if FLICKR_IMAGES else "")
print("CODE_DIR =", CODE_DIR, "\nWORK =", WORK, "\nSAVE =", SAVE)

## §1. Cài đặt
Cài thư viện (cần PyTorch có sẵn), clone repo của tác giả, vá 2 lỗi, sao `run_md_retrieval.py` vào repo. Chạy một lần.

In [ ]:
!python multidelete.py setup $COMMON

## §2. Chạy thử luồng (`smoke`)
Chạy với tập xóa nhỏ và vài bước huấn luyện để kiểm tra môi trường và luồng chạy. Kết quả smoke có tên riêng, không nằm trong bảng kết quả chính.

In [ ]:
!python multidelete.py flickr30k --profile smoke $COMMON

In [ ]:
!python multidelete.py nlvr2 --profile smoke $COMMON

## §3. Flickr30K, toàn bộ 
Gồm Original, MultiDelete, MD-ITC, Neg-Dr, ablation (-MD, -MKR, -UKR) và baseline (FineTune, NegGrad, DtD). Muốn chạy một phần, thêm `--only original,main` (xem README).

In [ ]:
!python multidelete.py flickr30k $COMMON

## §4. NLVR2, toàn bộ
Original, MultiDelete, FineTune, NegGrad. Thêm `--ablation` để chạy -MD/-MKR/-UKR.

In [ ]:
!python multidelete.py nlvr2 $COMMON

## §5. Demo định tính
Huấn luyện lại MultiDelete, lưu checkpoint rồi so sánh top-3 caption của mô hình gốc và mô hình đã quên.

In [ ]:
!python multidelete.py demo $COMMON

## §6. Xem kết quả

In [ ]:
import glob, pandas as pd
from IPython.display import display, Image as IPImage

for f in sorted(glob.glob(f"{SAVE}/flickr30k/results/summary_*.csv")):
    print("==", f); display(pd.read_csv(f).round(2))
f = f"{SAVE}/nlvr2/results/summary.csv"
if os.path.exists(f):
    print("==", f); display(pd.read_csv(f).round(2))
for f in sorted(glob.glob(f"{SAVE}/nlvr2/results/summary.png")) + sorted(glob.glob(f"{SAVE}/flickr30k/demo/*.png")):
    print(f); display(IPImage(f))